# Linked Data Structures

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Build a linked list from nodes allocated with malloc, each pointing to the next, and insert, remove and free nodes safely.
- Use linked lists as stacks and queues, and doubly linked lists to move both ways.
- Store and find values in a binary search tree, and print them in order with a recursive walk.

**In this notebook**, one idea at a time (run each cell before moving on):
1. A node points to the next one
2. Walking the list
3. Building a list with malloc
4. Freeing a list
5. Adding at the end
6. Removing a node
7. Beyond simple lists

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. A node points to the next one

A **linked list** is a chain of **nodes**. Each node holds a value and a pointer to the next node; the last one points to `NULL`. The struct refers to itself, so it needs a name before the `typedef` ends:

```c
typedef struct Node {
    int value;
    struct Node *next;
} Node;
```

Below, three nodes are created as ordinary local variables and linked by hand.

In [ ]:
#include <stdio.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    printf("%d -> %d -> %d\n", a.value, a.next->value, a.next->next->value);
    printf("after c comes %s\n", c.next == NULL ? "NULL" : "another node");
    return 0;
}

🧪 **Try it:** insert a node `x` with value 6 between `a` and `b` by changing only pointers: `x.next = &b; a.next = &x;`.

## 2. Walking the list

To visit every node, start a pointer at the head and follow `next` until it becomes `NULL`:

```c
for (const Node *p = head; p != NULL; p = p->next) { … }
```

There's no index: reaching the 100th node means following 100 pointers.

In [ ]:
#include <stdio.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

void print_list(const Node *head) {
    for (const Node *p = head; p != NULL; p = p->next) {
        printf("%d -> ", p->value);
    }
    printf("NULL\n");
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    print_list(&a);
    print_list(&b);
    print_list(NULL);
    return 0;
}

🧪 **Try it:** print only the values greater than 4.

### ✏️ Exercise: The length of a list

Count the nodes by following `next` until NULL. The test builds 7 → 5 → 3 with nodes on the stack, so nothing needs freeing.

In [ ]:
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_length(const Node *head) {
    // TODO: walk with a pointer until NULL
    return 0;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("three nodes", list_length(&a), 3);
    check_int("from the middle", list_length(&b), 2);
    check_int("empty", list_length(NULL), 0);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_length(const Node *head) {
    int n = 0;
    for (const Node *p = head; p != NULL; p = p->next) {
        n++;
    }
    return n;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("three nodes", list_length(&a), 3);
    check_int("from the middle", list_length(&b), 2);
    check_int("empty", list_length(NULL), 0);
    return 0;
}
```

</details>

### ✏️ Exercise: The sum of a list

Add up the values of every node.

In [ ]:
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_sum(const Node *head) {
    // TODO: walk the list adding p->value
    return 0;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("7 + 5 + 3", list_sum(&a), 15);
    check_int("empty", list_sum(NULL), 0);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
typedef struct Node {
    int value;
    struct Node *next;
} Node;

int list_sum(const Node *head) {
    int sum = 0;
    for (const Node *p = head; p != NULL; p = p->next) {
        sum += p->value;
    }
    return sum;
}

int main(void) {
    Node c = {3, NULL};
    Node b = {5, &c};
    Node a = {7, &b};
    check_int("7 + 5 + 3", list_sum(&a), 15);
    check_int("empty", list_sum(NULL), 0);
    return 0;
}
```

</details>

## 3. Building a list with malloc

Real lists grow while the program runs, so each node comes from `malloc`. Adding at the **head** is the simplest: the new node points to the old head, and becomes the new head. It takes the same time however long the list is. The slides show the list growing and being freed.

#### 🎞️ Building and freeing a linked list

![Pushing 3, 5 and 7 at the head gives 7 → 5 → 3 → NULL; freeing saves head->next before freeing each node](linked-list-steps.svg)

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = head;      // the new node points to the old head
    return n;            // and becomes the new head
}

int main(void) {
    Node *head = NULL;
    for (int v = 3; v <= 7; v += 2) {
        head = push(head, v);
    }
    for (Node *p = head; p != NULL; p = p->next) {
        printf("%d ", p->value);
    }
    printf("\n");
    while (head != NULL) {           // free every node
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}

🧪 **Try it:** why is the list printed in the opposite order of the pushes? Then forget to store the result: `push(head, v);` alone. What does the list hold now?

## 4. Freeing a list

Every node must be freed, one by one. The order matters: **save `head->next` before `free(head)`**, because after the free the node can't be read anymore, `next` included.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = head;
    return n;
}

int free_list(Node *head) {
    int freed = 0;
    while (head != NULL) {
        Node *next = head->next;   // save it first
        free(head);
        head = next;
        freed++;
    }
    return freed;
}

int main(void) {
    Node *head = NULL;
    for (int i = 0; i < 5; i++) {
        head = push(head, i);
    }
    printf("freed %d nodes\n", free_list(head));
    return 0;
}

🧪 **Try it:** change the loop to `free(head); head = head->next;` and run: the memory check reports the read after free.

## 5. Adding at the end

To add at the **end**, walk to the last node (the one whose `next` is `NULL`) and link the new node there. Walking makes it slower on long lists; keeping a `tail` pointer to the last node avoids the walk. An empty list is a special case: the new node becomes the head.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *append(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = NULL;
    if (head == NULL) {
        return n;                   // the list was empty
    }
    Node *last = head;
    while (last->next != NULL) {
        last = last->next;
    }
    last->next = n;
    return head;
}

int main(void) {
    Node *head = NULL;
    for (int v = 1; v <= 4; v++) {
        head = append(head, v * 10);
    }
    for (Node *p = head; p != NULL; p = p->next) {
        printf("%d ", p->value);
    }
    printf("\n");
    while (head != NULL) {
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}

🧪 **Try it:** compare with the push step: now the order is the order of the calls. Remove the `if (head == NULL)` case and run: what goes wrong with the first value?

## 6. Removing a node

To remove a node, make the node **before** it point to the node **after** it, then free it. Keep a `prev` pointer as you walk. Removing the head is the special case: the head moves to the second node.

In [ ]:
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) {
        exit(1);
    }
    n->value = value;
    n->next = head;
    return n;
}

Node *remove_value(Node *head, int value) {
    Node *prev = NULL;
    for (Node *p = head; p != NULL; prev = p, p = p->next) {
        if (p->value == value) {
            if (prev == NULL) {
                head = p->next;        // removing the head
            } else {
                prev->next = p->next;  // skip over p
            }
            free(p);
            return head;
        }
    }
    return head;                       // not found: nothing changes
}

void print_list(const Node *head) {
    for (const Node *p = head; p != NULL; p = p->next) {
        printf("%d ", p->value);
    }
    printf("\n");
}

int main(void) {
    Node *head = NULL;
    for (int v = 1; v <= 5; v++) {
        head = push(head, v);
    }
    print_list(head);
    head = remove_value(head, 3);
    head = remove_value(head, 5);
    head = remove_value(head, 42);
    print_list(head);
    while (head != NULL) {
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}

🧪 **Try it:** remove 1, the last node. Then remove every node one by one until the list is empty.

## 7. Beyond simple lists

The same idea builds other structures:

- a **doubly linked list** adds a `prev` pointer, so you can walk both ways and remove a node without searching for the one before it;
- a **stack** is a list where you push and pop at the head;
- a **queue** adds at the tail and removes at the head;
- a **binary tree** gives each node two pointers, `left` and `right`.

In [ ]:
#include <stdio.h>

typedef struct Tree {
    int value;
    struct Tree *left;
    struct Tree *right;
} Tree;

int sum(const Tree *t) {
    if (t == NULL) {
        return 0;
    }
    return t->value + sum(t->left) + sum(t->right);
}

int main(void) {
    Tree leaf1 = {2, NULL, NULL};
    Tree leaf2 = {9, NULL, NULL};
    Tree root = {5, &leaf1, &leaf2};
    printf("sum of the tree: %d\n", sum(&root));
    return 0;
}

🧪 **Try it:** add a node with value 4 under `leaf1` and check the new sum.

## In short

A **linked list** is a chain of nodes allocated one by one, each holding a value and a pointer to the next node; the last one points to `NULL`. Adding or removing a node only changes a few pointers, however long the list is.

```c
#include <stdio.h>
#include <stdlib.h>

typedef struct Node {
    int value;
    struct Node *next;
} Node;

Node *push(Node *head, int value) {
    Node *n = malloc(sizeof *n);
    if (n == NULL) exit(1);
    n->value = value;
    n->next = head;
    return n;
}

int main(void) {
    Node *head = NULL;
    for (int i = 1; i <= 3; i++) head = push(head, i);
    for (Node *p = head; p != NULL; p = p->next) printf("%d ", p->value);
    while (head != NULL) {
        Node *next = head->next;
        free(head);
        head = next;
    }
    return 0;
}
```

With a `prev` pointer too, the list is **doubly linked**; with `left` and `right`, nodes form a **tree**.

**Watch out:** save `head->next` before `free(head)`: after the free, the node can't be read.

## Readings

- [Lecture 5: Data Structures](https://cs50.harvard.edu/x/2026/notes/5/#data-structures) (CS50x (Harvard), lecture notes, 25 min): From "Data Structures" to "Trees": queues and stacks, why arrays are hard to grow, linked lists drawn as boxes and arrows, and binary search trees. Hash tables and tries come after, for later.
- [Self-Referential structs](https://beej.us/guide/bgc/html/split/structs-ii-more-fun-with-structs.html#self-referential-structs) (Beej's Guide to C Programming, chapter 20, 4 min): Optional: why a struct can contain a pointer to its own type (but not itself), with a small linked list built and walked.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **RPN Calculator** · Easy
- **Printer Queue** · Medium
- **Undo and Redo** · Medium
- **BST Word Counter** · Hard
- Test: **Line Editor** · Hard